# LIVE Reliability Patterns

## What this notebook proves
- Managed LIVE streams can receive realtime events with retry/backpressure settings.
- Shared/fanout LIVE subscriptions can be started explicitly.
- You get a concrete event payload and a PASS/FAIL smoke summary.

## Prerequisites
- SurrealDB running at `ws://localhost:8000/rpc`
- Credentials `root` / `root`


In [ ]:
import asyncio
from surrealengine import Document, StringField, create_connection

class FeedItem(Document):
    text = StringField()

    class Meta:
        collection = "feed_items_live"


In [ ]:
conn = create_connection(
    url="ws://localhost:8000/rpc",
    namespace="demo",
    database="live_demo",
    username="root",
    password="root",
    make_default=True,
)
await conn.connect()
await FeedItem.create_table()


In [ ]:
# Option 1: managed full-stream watcher
async def consume_live(limit=5):
    seen = 0
    async for evt in FeedItem.objects.live(
        action=["CREATE", "UPDATE"],
        queue_maxsize=500,
        backpressure_policy="drop_oldest",
        retry_limit=5,
        initial_delay=0.5,
        backoff=2.0,
    ):
        print(evt.action, evt.id)
        seen += 1
        if seen >= limit:
            break

# Run in task in app startup; cancel on shutdown.


In [ ]:
# Option 2: explicit fanout pattern
# start once
qid = await FeedItem.objects.start_live()

async def consumer(name):
    async for evt in FeedItem.objects.subscribe_to_live(qid, queue_maxsize=250):
        print(name, evt.action, evt.id)

# multiple consumers can share same qid
# await asyncio.gather(consumer('A'), consumer('B'))


In [ ]:
# Smoke test for managed stream: subscribe, write, receive one event
stream = FeedItem.objects.live(
    action=["CREATE"],
    queue_maxsize=100,
    backpressure_policy="drop_oldest",
    retry_limit=3,
    initial_delay=0.2,
    backoff=2.0,
)

event_task = asyncio.create_task(anext(stream))
await asyncio.sleep(0.2)
created = await FeedItem.objects.create(text="live smoke event")
received = await asyncio.wait_for(event_task, timeout=10)

print("Created:", created.id)
print("Received event:", received.action, received.id)

await stream.aclose()


In [ ]:
await conn.disconnect()


In [ ]:
# Validate fanout start and clean shutdown
print("Shared qid:", qid)
if not qid:
    raise AssertionError("Expected non-empty live query id")

await conn.client.kill(qid)
print("Killed shared qid successfully")
print("ALL LIVE SMOKE CHECKS PASSED")
